# 📈 JP Backtest — Complete System Architecture Walkthrough

Welcome to the **Interactive Architecture Guide** for the Japan Stock Backtesting Application. 

This single notebook will walk you linearly through the entire lifecycle of a daily stock price from the exact moment it is pulled from the stock market to the exact moment it visually renders on the dashboard for the end-user.

You won't just read about the architecture—**you will actively run the code that powers it**. We will simulate the heavy lifting of the message queue, the horizontal scaling of Apache Spark, and the real-time inference of the AWS Machine Learning model.

***Note:*** *You can actively run the code cells below by clicking on them and pressing `Shift + Enter`.*

In [ ]:
# Run this cell first to ensure all required dependencies for the simulation are installed in your environment.
!pip install yfinance pandas numpy quantstats xgboost pyspark scikit-learn

---
## Phase 1: The Producer (Data Ingestion via `yfinance`)

The journey begins at the edge of our system. A Python microservice acts as the **Producer**. Its only job is to constantly poll the Yahoo Finance API for the latest daily Open, High, Low, Close, and Volume (OHLCV) records for Japanese blue-chip stocks.

Let's simulate the Producer actively pulling a live tick.

In [ ]:
import yfinance as yf
import pandas as pd

ticker = '7203.T'  # Toyota Motor Corp
print(f"📡 PRODUCER: Fetching live data for {ticker}...")

# Fetch the last 10 days of active market data
history = yf.download(ticker, period='10d', progress=False)

if isinstance(history.columns, pd.MultiIndex):
    history.columns = history.columns.droplevel(1)

print("✅ PRODUCER: Successfully grabbed active market ticks:")
display(history[['Open', 'High', 'Low', 'Close', 'Volume']].tail(3))

---
## Phase 2: The Message Queue (BlazingMQ & The 'Slow Consumer' Problem)

Now that the Producer has the data, it needs to save it to our database. 

**Why not just save it directly to the database?**
If the market opens and thousands of ticker updates flow in per second, writing them to a massive database sequentially requires heavy network calls and disk I/O. If the database gets slow, your Producer gets stuck waiting, causing it to crash or miss live stock prices entirely. 

To solve this, we place **BlazingMQ** (Bloomberg's high-performance open-source message queue) between the Producer and the Database (Consumer). The Producer instantly drops the data into the in-memory queue in 0.01 seconds and goes back to fetching more stocks. A separate Consumer slowly drains the queue at its own safe pace.

Let's simulate this asynchronous decoupled buffering below:

In [ ]:
import time
import queue
import threading

# Simulating 5 rapid data ticks fetched from the live market
market_ticks = [2800.0, 2801.5, 2802.0, 2799.0, 2805.0]

blazing_mq = queue.Queue()  # Simulating the BlazingMQ Cluster

def blazingmq_producer():
    start_time = time.time()
    for price in market_ticks:
        blazing_mq.put(price) # Drop message onto the MQ
        print(f"[{time.strftime('%H:%M:%S')}] 🚀 PRODUCER: Instantly queued payload: {price}")
        time.sleep(0.01) # Ultra-fast market fetch
        
    print(f"✅ PRODUCER: Finished all my work in {time.time() - start_time:.2f} seconds! I am completely free to do other things.")

def blazingmq_consumer():
    while not blazing_mq.empty():
        price = blazing_mq.get()
        # Simulates a heavy, slow database insertion
        time.sleep(1.0) 
        print(f"[{time.strftime('%H:%M:%S')}] 💾 CONSUMER: Slowly, but safely saved payload to Database: {price}")
        blazing_mq.task_done()
    print("✅ CONSUMER: Automatically finished draining the queue!")

print("--- Starting Decoupled MQ Integration ---")
producer_thread = threading.Thread(target=blazingmq_producer)
consumer_thread = threading.Thread(target=blazingmq_consumer)

producer_thread.start()
time.sleep(0.1)
consumer_thread.start()

producer_thread.join()
consumer_thread.join()

---
## Phase 3: The Data Platform (Databricks, Apache Spark, & Delta Lake)

When the Consumer drains BlazingMQ, it saves the payloads permanently into a **Databricks Delta Lake**.

**Why Databricks instead of PostgreSQL?**
Financial Feature Engineering requires running intense math (like a 200-day rolling average) across millions of rows spanning thousands of stocks. A standard relational database scales *vertically* (one single massive computer hard-drive), meaning it will eventually freeze on huge queries. Databricks orchestrates **Apache Spark**, which scales *horizontally*, splitting the math dataset into a thousand smaller chunks across hundreds of separate computers simultaneously in RAM.

Let's spin up a local PySpark cluster right now to simulate distributed feature engineering:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, avg

print("Spinning up local Apache Spark cluster...")
spark = SparkSession.builder.appName("TradingBacktestApp").getOrCreate()

spark_df = spark.createDataFrame(
    history.reset_index()[['Date', 'Close']].assign(Ticker='7203.T')
)

# The Window explicitly groups the math by Ticker, so Spark knows exactly how to distribute the load across multiple computers
windowSpec = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-4, 0) # 5-day Simple Moving Average

# Calculate the average concurrently across the distributed window
df_features = spark_df.withColumn("SMA_5", avg(col("Close")).over(windowSpec))

print("\nDistributed Apache Spark Feature Engineering Complete:")
df_features.show(5)

---
## Phase 4: Machine Learning (AWS SageMaker)

The clean "feature data" calculated by Spark (SMA, RSI) forms the ground truth for our active trading application. We now pipe this data securely out of Databricks and straight into **AWS SageMaker**.

SageMaker hosts our **XGBoost Classifier Model**. 
- In **Training Mode**, we feed it 10 years of feature data to find complex, non-linear correlations predicting stock drift.
- In **Inference Mode**, we pass it today's newly calculated Databricks row, and it instantly responds with a highly-confident `BUY`, `SELL`, or `HOLD` flag.

Let's actively simulate training an algorithmic trading classifier right now:

In [ ]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import accuracy_score

# 1. Synthesize 1000 days of Feature Data for Training (RSI, SMA, Momentum)
print("🧠 AWS SageMaker: Piping historical Delta Lake feature data into modeling engine...")
np.random.seed(42)
X_train = np.random.rand(1000, 3) # 3 Features

# Target: '1' if the stock will go up 5% in the next 20 days, '0' otherwise.
y_train = (X_train[:, 0] + X_train[:, 1] > 1.2).astype(int) 
X_test = np.random.rand(200, 3)
y_test = (X_test[:, 0] + X_test[:, 1] > 1.2).astype(int)

# 2. Train the XGBoost Backtesting Model
model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(f"\n✅ AWS SageMaker: Training complete! Simulation Accuracy: {accuracy_score(y_test, preds):.2%}")

print("\n🔮 Inference: If we pass today's live stock data to the SageMaker InvokeEndpoint API...")
live_prediction = model.predict(np.array([[0.8, 0.9, 0.2]]))
decision = "BUY" if live_prediction[0] == 1 else "HOLD/SELL"
print(f"➡️ SageMaker Responds: {decision}")

---
## Phase 5: The Middleware (Java Spring Boot API)

With the Data perfected by **Databricks** and the Intelligence generated by **AWS SageMaker**, we need to safely expose this state to the open internet. We do not let Web/Mobile apps query our data lake or models directly!

Our **Java Spring Boot API** acts as the secure middleman block of the application. It fetches the aggregated state, caches it in memory, handles enterprise security routing (CORS/Authentication), and serves strictly-typed JSON through REST standard endpoints (`@GetMapping("/api/stocks")`).

---
## Phase 6: The Frontend (React + Vite SPA)

The final stop is the user's browser. Our dynamic **React 18** frontend fetches the flawless JSON output produced by the Java backend using `axios`.

Because all the heavy-lifting (the yFinance polling, the Spark distributed window math, the XGBoost Decision Trees) occurred upstream, the React frontend is extraordinarily fast. It simply Maps the JSON array into beautifully themed CSS tracking cards, attaching native CSS `onHover` tooltip layers for user explanation.

### Complete Flow Conclusion:
**`yFinance Trigger ➔ BlazingMQ Buffer ➔ Spark Feature Engineering ➔ SageMaker AI Inference ➔ Java Spring Boot Security ➔ React Dashboard Rendering.`**